# 02 — EfficientNet-B3 Training
Train the EfficientNet-B3 classifier on the brain tumor MRI dataset.

In [ ]:
import sys
from pathlib import Path
sys.path.append('../src')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from classification.efficientnet_classifier import EfficientNetB3Classifier, CLASSES
from utils.visualization import plot_training_curves, plot_confusion_matrix

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = Path('../data/processed')
MODEL_OUT = Path('../models/efficientnet_b3_brain_tumor.pt')
IMG_SIZE = 300
EPOCHS = 50
BATCH_SIZE = 32
LR = 1e-3

print(f'Device: {DEVICE}')

In [ ]:
# Data loaders
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(DATA_DIR / 'train', transform=train_tf)
val_ds   = datasets.ImageFolder(DATA_DIR / 'val',   transform=val_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# Model, loss, optimizer
model     = EfficientNetB3Classifier(num_classes=len(CLASSES), pretrained=True).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
print('Model ready.')

In [ ]:
train_losses, val_accs = [], []
best_acc = 0.0

for epoch in range(EPOCHS):
    # Train
    model.train()
    epoch_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        correct += out.argmax(1).eq(labels).sum().item()
        total   += labels.size(0)

    train_losses.append(epoch_loss / len(train_loader))

    # Validate
    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            out = model(imgs.to(DEVICE))
            v_correct += out.argmax(1).cpu().eq(labels).sum().item()
            v_total   += labels.size(0)
    val_acc = 100 * v_correct / v_total
    val_accs.append(val_acc)

    print(f'Epoch {epoch+1:02d}/{EPOCHS}  Loss: {train_losses[-1]:.4f}  Val Acc: {val_acc:.2f}%')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), MODEL_OUT)
        print(f'  ✅ Saved best model  ({val_acc:.2f}%)')

    scheduler.step()

print(f'\nBest Val Accuracy: {best_acc:.2f}%')

In [ ]:
plot_training_curves(train_losses, val_accs, save_path='../docs/training_curves.png')